# Phase 3 — QLoRA fine-tune Qwen3-8B

Thin driver. All logic lives in the repo (`scripts/train_student.py`, `src/pii/eval.py`) so it is
version-controlled and diffable; a notebook that holds the logic cannot be reviewed or reverted.

**Session settings:** Accelerator `GPU T4 x2`, Internet `On`, Persistence `Variables and Files`.
**Secrets required:** `HF_TOKEN`, `WANDB_API_KEY` — both must be *attached* to this notebook.
**Dataset required:** `pii-distillation-data`, added via Add Input.

Run cells 1–4 once per session, then one cell from section 5.

> For the two ~5h runs use **Save Version → Save & Run All**, not the interactive session:
> it runs headless on Kaggle's servers, so a closed laptop is irrelevant.

## 1 · Repo and dependencies

In [ ]:
!git clone -q --branch phase-3 https://github.com/eren-o23/model-distillation-pipeline.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -1

# torch is NOT in requirements-train.txt: Kaggle's image ships a CUDA build matched to its driver,
# and replacing it costs 2.5GB and may not load.
!pip install -q -r requirements-train.txt
!python -c "import torch, transformers, peft, bitsandbytes; print('torch', torch.__version__, '| cuda', torch.version.cuda)"

## 2 · Data

`data/*` is gitignored, so the clone arrives empty. Symlinking the attached dataset into the repo's
`data/` keeps `DATA_DIR` and `load_split()` working unchanged — no paths threaded through the code.

In [ ]:
from pathlib import Path

# The mount path is discovered, not hardcoded. Kaggle slugifies the dataset title, so the folder name
# need not match what you typed, and any subfolder structure from the upload is preserved.
ROOT_IN = Path('/kaggle/input')
DST = Path('/kaggle/working/repo/data')
DST.mkdir(exist_ok=True)

found = {p.name: p for p in ROOT_IN.rglob('*.jsonl')}
if not found:
    listing = '\n  '.join(str(p) for p in sorted(ROOT_IN.rglob('*'))[:40]) or '(nothing mounted)'
    raise SystemExit(
        f'No .jsonl files under /kaggle/input. What is actually mounted:\n  {listing}\n'
        'Add the dataset via Add Input, then Run > Restart session.'
    )

for name in ('train_sft.jsonl', 'val_sft.jsonl', 'val.jsonl'):
    target = found.get(name)
    assert target, f'{name} not found. Mounted .jsonl files: {sorted(found)}'
    link = DST / name
    link.unlink(missing_ok=True)
    link.symlink_to(target)
    print(f'{name:20} {target.stat().st_size / 1e6:7.1f} MB   <- {target.parent}')

# test.jsonl is deliberately absent — it stays sealed until Phase 4, and load_split('test') raises
# without allow_test=True, which appears nowhere in the Phase 3 code.
assert 'test.jsonl' not in found, 'test split must not be uploaded — it stays sealed until Phase 4'
assert not (DST / 'test.jsonl').exists(), 'test split must not be present during Phase 3'

## 3 · Secrets

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

s = UserSecretsClient()
os.environ['HF_TOKEN'] = s.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = s.get_secret('WANDB_API_KEY')

# The 16GB base model must not land in /kaggle/working, which is capped at 20GB and is also where
# checkpoints go. /kaggle/temp is scratch with far more room.
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('secrets loaded:', all(os.environ.get(k) for k in ('HF_TOKEN', 'WANDB_API_KEY')))

## 4 · Preflight

Everything here fails in seconds if it is going to fail at all. The point is to never discover a
sm_75 or prompt-rendering problem five hours into a run.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!python -m pytest tests/ -q

## 5 · Runs

In order. The smoke test first — it reports **measured s/step**, which is what decides whether the
real runs stay single-GPU or move to `torchrun --nproc_per_node=2`.

In [ ]:
# Smoke test — ~20 min, most of it model download. Proves the loop and reports s/step.
!python -u scripts/train_student.py --rank 8 --limit 256 --epochs 1 --eval-n 32 --no-push

In [ ]:
# Baselines — untuned Qwen3-8B. Answers "how much did fine-tuning buy over just prompting?"
!python -u scripts/train_student.py --baseline short
!python -u scripts/train_student.py --baseline teacher

In [ ]:
# Config A. Add --resume if a session was killed part way.
!python -u scripts/train_student.py --rank 8

In [ ]:
# Config B — identical except rank (alpha tracks it at 2r, so capacity is the only variable).
!python -u scripts/train_student.py --rank 32

In [ ]:
# Final: the winning adapter over all 1,000 val rows, for the headline number.
!python -u scripts/train_student.py --eval-adapter erenrosman/pii-qwen3-8b-lora-r8 --eval-n 0

## 6 · Collect

`reports/raw/phase3/*.json` is what `scripts/write_phase3.py` turns into the report, so these files
must come back off Kaggle. They are small.

In [ ]:
!mkdir -p /kaggle/working/out && cp -r /kaggle/working/repo/reports/raw/phase3 /kaggle/working/out/
!ls -la /kaggle/working/out/phase3/